# Short-horizon rebuild notebook (1h, 6h, 12h, 24h)

This notebook rebuilds the eight-model comparison from the original project using the same overall methods, but with shorter forecast horizons.

The model set is:

1. Elastic Net  
2. Random Forest  
3. LSTM  
4. CNN–GRU–LSTM  
5. GraphWaveNet  
6. GraphWaveNet–GRU  
7. GraphWaveNet–LSTM  
8. GraphWaveNet–GRU–LSTM  

The methodological goal is to keep the project aligned with the original notebooks:

- same split dates
- same station set
- same feature channels
- same train-only scaling
- same strict sliding-window logic
- same evaluation style in original traffic-flow units

The only forecasting change is:

- old setup: `OUT_LEN = 72`, report `12, 24, 48, 72`
- new setup: `OUT_LEN = 24`, report `1, 6, 12, 24`


In [1]:
from pathlib import Path
import copy
import gc
import json
import random
import time

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import MultiTaskElasticNet
from sklearn.ensemble import RandomForestRegressor
from joblib import Parallel, delayed

from IPython.display import display


def set_seed(seed: int = 42, deterministic: bool = True):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


SEED = 42
set_seed(SEED, deterministic=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# -------------------------
# Raw files
# -------------------------
TRAFFIC_CSV = Path("cleaned_traffic_data.csv")
META_XLSX   = Path("pems_output.xlsx")

# -------------------------
# Existing strict artifact from the original project
# -------------------------
BASE_STRICT_DATASET = Path("artifacts/pems_graph_dataset_strict.npz")

# -------------------------
# Split boundaries (same as original project)
# -------------------------
TRAIN_END = pd.Timestamp("2024-11-15 23:59:59")
VAL_END   = pd.Timestamp("2024-11-30 23:59:59")

# -------------------------
# Short-horizon setup
# -------------------------
IN_LEN = 24
OUT_LEN = 24
EVAL_HORIZONS = [1, 6, 12, 24]

assert max(EVAL_HORIZONS) <= OUT_LEN, "The largest evaluation horizon must be <= OUT_LEN."

SHORT_TAG = "short_h1_6_12_24"

OUT_DIR = Path("artifacts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SHORT_DATASET = OUT_DIR / f"pems_graph_dataset_strict_{SHORT_TAG}.npz"
RESULTS_SUMMARY_SHORT = OUT_DIR / f"results_summary_{SHORT_TAG}.csv"

print("BASE_STRICT_DATASET:", BASE_STRICT_DATASET)
print("SHORT_DATASET:", SHORT_DATASET)
print("RESULTS_SUMMARY_SHORT:", RESULTS_SUMMARY_SHORT)

Torch: 2.1.1+cu121
Device: cpu
BASE_STRICT_DATASET: artifacts/pems_graph_dataset_strict.npz
SHORT_DATASET: artifacts/pems_graph_dataset_strict_short_h1_6_12_24.npz
RESULTS_SUMMARY_SHORT: artifacts/results_summary_short_h1_6_12_24.csv


## Rebuild the strict dataset for 24-step forecasting

This is the key short-horizon conversion step.

To stay methodologically faithful to the original project, we do not rebuild the whole cleaned dataset from scratch here.  
Instead, we:

- load the already-built strict artifact from the original notebook
- keep the same `X`, `Y`, adjacency, stations, timestamps, and train-only statistics
- recompute only the valid start indices and split assignments for `OUT_LEN = 24`

That way, the study changes only in the forecast length and evaluation horizons.

In [ ]:
assert BASE_STRICT_DATASET.exists(), (
    f"Missing {BASE_STRICT_DATASET}. "
    "Run the original notebook once so the strict base artifact exists."
)

base = np.load(BASE_STRICT_DATASET, allow_pickle=True)

X_base = base["X"].astype(np.float32)
Y_base = base["Y"].astype(np.float32)
A_base = base["A"].astype(np.float32)
stations_base = base["stations"]
timestamps_base = pd.to_datetime(base["timestamps"])

flow_mean_base = base["flow_mean"].astype(np.float32)
flow_std_base  = base["flow_std"].astype(np.float32)
speed_mean_base = base["speed_mean"].astype(np.float32)
speed_std_base  = base["speed_std"].astype(np.float32)

base_in_len = int(np.array(base["in_len"]).item())
print("Base strict artifact loaded.")
print("Base X:", X_base.shape, "Base Y:", Y_base.shape)
print("Base IN_LEN:", base_in_len)

assert base_in_len == IN_LEN, (
    f"Expected base IN_LEN={IN_LEN}, found {base_in_len}. "
    "This notebook assumes the original project used 24 hours of input."
)

T_total = X_base.shape[0]
max_start = T_total - (IN_LEN + OUT_LEN) + 1
starts = np.arange(max_start, dtype=np.int32)

out_start_times = timestamps_base[starts + IN_LEN]
out_end_times   = timestamps_base[starts + IN_LEN + OUT_LEN - 1]

train_starts_short = starts[out_end_times <= TRAIN_END]
val_starts_short   = starts[(out_start_times > TRAIN_END) & (out_end_times <= VAL_END)]
test_starts_short  = starts[out_start_times > VAL_END]

print("Short-horizon strict window counts:")
print("train:", len(train_starts_short))
print("val:  ", len(val_starts_short))
print("test: ", len(test_starts_short))

np.savez_compressed(
    SHORT_DATASET,
    X=X_base.astype(np.float32),
    Y=Y_base.astype(np.float32),
    A=A_base.astype(np.float32),
    stations=stations_base,
    timestamps=np.array(timestamps_base.astype("datetime64[ns]")),
    train_starts=train_starts_short,
    val_starts=val_starts_short,
    test_starts=test_starts_short,
    in_len=np.array([IN_LEN], dtype=np.int32),
    out_len=np.array([OUT_LEN], dtype=np.int32),
    flow_mean=flow_mean_base,
    flow_std=flow_std_base,
    speed_mean=speed_mean_base,
    speed_std=speed_std_base,
)

print("Saved short-horizon strict dataset to:", SHORT_DATASET)

## Load the short-horizon shared tensors

We keep the same shared representation used in the original notebooks:

- `X` with shape `(T, N, F)`
- `Y` with shape `(T, N)`

The feature channels remain:

1. flow  
2. speed  
3. hour_sin  
4. hour_cos  
5. dow_sin  
6. dow_cos  

Only the output sequence length changes from `72` to `24`.

In [2]:
assert SHORT_DATASET.exists(), f"Missing {SHORT_DATASET}. Run the rebuild cell above first."

data = np.load(SHORT_DATASET, allow_pickle=True)

X = data["X"].astype(np.float32)          # (T, N, F)
Y = data["Y"].astype(np.float32)          # (T, N)
A = data["A"].astype(np.float32)          # (N, N)

stations = data["stations"]
timestamps = pd.to_datetime(data["timestamps"])

train_starts = data["train_starts"].astype(np.int64)
val_starts   = data["val_starts"].astype(np.int64)
test_starts  = data["test_starts"].astype(np.int64)

IN_LEN  = int(np.array(data["in_len"]).item())
OUT_LEN = int(np.array(data["out_len"]).item())

flow_mean  = data["flow_mean"].astype(np.float32)
flow_std   = data["flow_std"].astype(np.float32)
speed_mean = data["speed_mean"].astype(np.float32)
speed_std  = data["speed_std"].astype(np.float32)

flow_std  = np.maximum(flow_std, 1e-6).astype(np.float32)
speed_std = np.maximum(speed_std, 1e-6).astype(np.float32)

T, N, Fdim = X.shape
print("X:", X.shape, "(T,N,F)")
print("Y:", Y.shape, "(T,N)")
print("A:", A.shape)
print("IN_LEN / OUT_LEN:", IN_LEN, OUT_LEN)
print("Stations:", N)
print("train / val / test starts:", len(train_starts), len(val_starts), len(test_starts))


def time_encoding(dt_index: pd.DatetimeIndex) -> np.ndarray:
    hours = dt_index.hour.values
    dow   = dt_index.dayofweek.values
    hour_sin = np.sin(2 * np.pi * hours / 24.0)
    hour_cos = np.cos(2 * np.pi * hours / 24.0)
    dow_sin  = np.sin(2 * np.pi * dow / 7.0)
    dow_cos  = np.cos(2 * np.pi * dow / 7.0)
    return np.stack([hour_sin, hour_cos, dow_sin, dow_cos], axis=1).astype(np.float32)


TF_all = time_encoding(pd.DatetimeIndex(timestamps))  # (T, 4)

# Scale only the continuous sensor channels using train-only stats saved in the artifact.
X_scaled = X.copy()
X_scaled[:, :, 0] = (X_scaled[:, :, 0] - flow_mean[None, :]) / flow_std[None, :]
X_scaled[:, :, 1] = (X_scaled[:, :, 1] - speed_mean[None, :]) / speed_std[None, :]

Y_scaled = (Y - flow_mean[None, :]) / flow_std[None, :]

# Conv-style layout reused in the original notebooks: (F, N, T)
X_fnt = np.transpose(X_scaled, (2, 1, 0)).copy()

print("TF_all:", TF_all.shape)
print("X_fnt:", X_fnt.shape)
print("Scaled target mean/std:", float(Y_scaled.mean()), float(Y_scaled.std()))

X: (2208, 1821, 6) (T,N,F)
Y: (2208, 1821) (T,N)
A: (1821, 1821)
IN_LEN / OUT_LEN: 24 24
Stations: 1821
train / val / test starts: 1057 337 721
TF_all: (2208, 4)
X_fnt: (6, 1821, 2208)
Scaled target mean/std: -781.403564453125 30704.76953125


## Shared dataset and dataloaders for all deep models

All deep models use the same sliding-window dataset:

- `x`: `(F, N, IN_LEN)`
- `y`: `(OUT_LEN, N)` in scaled flow units
- `tf`: `(OUT_LEN, 4)` future calendar features for the target steps

This matches the later clean sections of the original project.

In [3]:
class FastPemsWindowDataset(Dataset):
    def __init__(self, X_fnt, Y_scaled, TF_all, starts, in_len, out_len):
        self.X_fnt = X_fnt
        self.Y = Y_scaled
        self.TF = TF_all
        self.starts = np.asarray(starts, dtype=np.int64)
        self.in_len = int(in_len)
        self.out_len = int(out_len)

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, idx):
        s = int(self.starts[idx])
        x = self.X_fnt[:, :, s:s + self.in_len]                    # (F, N, IN)
        y = self.Y[s + self.in_len:s + self.in_len + self.out_len, :]  # (OUT, N)
        tf = self.TF[s + self.in_len:s + self.in_len + self.out_len, :] # (OUT, 4)

        return (
            torch.from_numpy(x).float(),
            torch.from_numpy(y).float(),
            torch.from_numpy(tf).float(),
        )


train_ds = FastPemsWindowDataset(X_fnt, Y_scaled, TF_all, train_starts, IN_LEN, OUT_LEN)
val_ds   = FastPemsWindowDataset(X_fnt, Y_scaled, TF_all, val_starts,   IN_LEN, OUT_LEN)
test_ds  = FastPemsWindowDataset(X_fnt, Y_scaled, TF_all, test_starts,  IN_LEN, OUT_LEN)

BATCH_SIZE = 8

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

xb, yb, tfb = next(iter(train_loader))
print("Batch x:", xb.shape)
print("Batch y:", yb.shape)
print("Batch tf:", tfb.shape)

Batch x: torch.Size([8, 6, 1821, 24])
Batch y: torch.Size([8, 24, 1821])
Batch tf: torch.Size([8, 24, 4])


## Shared evaluator and run-output utilities

All deep models are trained on scaled targets and evaluated in original traffic-flow units.

The evaluation rule is unchanged from the original notebooks:

- horizon `h` maps to output index `h - 1`
- MAE and RMSE are computed in original vehicles/hour units
- early stopping monitors average validation MAE across the selected horizons

This cell also writes a fresh short-horizon summary CSV so the new results do not overwrite the old 72-hour study.

In [4]:
flow_mean_t = torch.tensor(flow_mean, dtype=torch.float32, device=DEVICE).view(1, 1, -1)
flow_std_t  = torch.tensor(flow_std,  dtype=torch.float32, device=DEVICE).view(1, 1, -1)


def print_metrics(title: str, metrics: dict):
    print("\n" + title)
    for h in sorted(metrics.keys()):
        print(f"  {h:>3}h  MAE={metrics[h]['MAE']:.3f}  RMSE={metrics[h]['RMSE']:.3f}")


def avg_mae(metrics: dict) -> float:
    return float(np.mean([metrics[h]["MAE"] for h in metrics]))


@torch.inference_mode()
def eval_horizons_fast(model, loader, eval_horizons=EVAL_HORIZONS):
    model.eval()
    acc = {h: {"abs": 0.0, "sq": 0.0, "count": 0} for h in eval_horizons}

    for xb, yb, tfb in tqdm(loader, desc="Eval", leave=False):
        xb  = xb.to(DEVICE, non_blocking=True)
        yb  = yb.to(DEVICE, non_blocking=True)
        tfb = tfb.to(DEVICE, non_blocking=True)

        pred = model(xb, tfb)  # scaled (B, OUT, N)

        pred_u = pred * flow_std_t + flow_mean_t
        true_u = yb   * flow_std_t + flow_mean_t

        for h in eval_horizons:
            idx = h - 1
            err = pred_u[:, idx, :] - true_u[:, idx, :]
            acc[h]["abs"] += float(err.abs().sum())
            acc[h]["sq"]  += float((err ** 2).sum())
            acc[h]["count"] += err.numel()

    metrics = {}
    for h in eval_horizons:
        mae = acc[h]["abs"] / acc[h]["count"]
        rmse = (acc[h]["sq"] / acc[h]["count"]) ** 0.5
        metrics[h] = {"MAE": float(mae), "RMSE": float(rmse)}
    return metrics


def make_run_dir(model_name: str, tag: str = SHORT_TAG) -> Path:
    stamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    run_dir = Path("artifacts/runs") / f"{stamp}_{model_name}_{tag}"
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir


def save_json(path: Path, obj: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)


def save_metrics_files(run_dir: Path, split_name: str, metrics: dict):
    save_json(run_dir / f"{split_name}_metrics.json", metrics)
    rows = [{"horizon_h": h, "MAE": metrics[h]["MAE"], "RMSE": metrics[h]["RMSE"]} for h in sorted(metrics)]
    pd.DataFrame(rows).to_csv(run_dir / f"{split_name}_metrics.csv", index=False)


def append_results_summary(model_name: str, run_dir: Path, test_metrics: dict):
    row = {
        "timestamp": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        "model_name": model_name,
        "run_dir": str(run_dir),
        "tag": SHORT_TAG,
    }
    for h in EVAL_HORIZONS:
        row[f"test_MAE_{h}h"] = float(test_metrics[h]["MAE"])
        row[f"test_RMSE_{h}h"] = float(test_metrics[h]["RMSE"])

    df_new = pd.DataFrame([row])
    if RESULTS_SUMMARY_SHORT.exists():
        df_old = pd.read_csv(RESULTS_SUMMARY_SHORT)
        df = pd.concat([df_old, df_new], ignore_index=True)
    else:
        df = df_new

    df.to_csv(RESULTS_SUMMARY_SHORT, index=False)
    return RESULTS_SUMMARY_SHORT


@torch.inference_mode()
def collect_selected_horizon_predictions(model, loader, horizons=EVAL_HORIZONS):
    model.eval()
    pred_list = []
    true_list = []

    for xb, yb, tfb in tqdm(loader, desc="Collect preds", leave=False):
        xb  = xb.to(DEVICE, non_blocking=True)
        yb  = yb.to(DEVICE, non_blocking=True)
        tfb = tfb.to(DEVICE, non_blocking=True)

        pred = model(xb, tfb)
        pred_u = pred * flow_std_t + flow_mean_t
        true_u = yb   * flow_std_t + flow_mean_t

        pred_sel = []
        true_sel = []
        for h in horizons:
            idx = h - 1
            pred_sel.append(pred_u[:, idx, :].detach().cpu().numpy())
            true_sel.append(true_u[:, idx, :].detach().cpu().numpy())

        pred_sel = np.stack(pred_sel, axis=1)   # (B, H, N)
        true_sel = np.stack(true_sel, axis=1)   # (B, H, N)

        pred_list.append(pred_sel.astype(np.float32))
        true_list.append(true_sel.astype(np.float32))

    pred_all = np.concatenate(pred_list, axis=0)
    true_all = np.concatenate(true_list, axis=0)
    return pred_all, true_all


def train_deep_model(
    model_name: str,
    model: nn.Module,
    train_loader,
    val_loader,
    test_loader,
    epochs: int = 40,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    clip: float = 5.0,
    patience: int = 6,
    eval_every: int = 2,
    save_predictions: bool = True,
):
    run_dir = make_run_dir(model_name)
    print("Run dir:", run_dir)

    config = {
        "model_name": model_name,
        "seed": SEED,
        "in_len": IN_LEN,
        "out_len": OUT_LEN,
        "eval_horizons": EVAL_HORIZONS,
        "epochs": epochs,
        "lr": lr,
        "weight_decay": weight_decay,
        "clip": clip,
        "patience": patience,
        "eval_every": eval_every,
        "tag": SHORT_TAG,
    }
    save_json(run_dir / "config.json", config)

    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.SmoothL1Loss(beta=1.0)

    best_score = float("inf")
    best_state = None
    best_epoch = None
    bad_epochs = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        running = 0.0

        for xb, yb, tfb in tqdm(train_loader, desc=f"Train {epoch}/{epochs}", leave=False):
            xb  = xb.to(DEVICE, non_blocking=True)
            yb  = yb.to(DEVICE, non_blocking=True)
            tfb = tfb.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            pred = model(xb, tfb)
            loss = loss_fn(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()

            running += float(loss.item())

        row = {"epoch": epoch, "train_loss": running / max(len(train_loader), 1)}

        if (epoch % eval_every == 0) or (epoch == epochs):
            val_metrics = eval_horizons_fast(model, val_loader)
            val_avg = avg_mae(val_metrics)

            row["val_avg_MAE"] = float(val_avg)
            for h in EVAL_HORIZONS:
                row[f"val_MAE_{h}h"] = float(val_metrics[h]["MAE"])
                row[f"val_RMSE_{h}h"] = float(val_metrics[h]["RMSE"])

            print(f"\nEpoch {epoch}: train_loss={row['train_loss']:.6f}  val_avg_MAE={val_avg:.3f}")
            print_metrics("Validation", val_metrics)

            if val_avg < best_score:
                best_score = float(val_avg)
                best_epoch = int(epoch)
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                torch.save(best_state, run_dir / "best.pt")
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    print(f"\nEarly stopping. Best val_avg_MAE={best_score:.3f} at epoch {best_epoch}.")
                    history.append(row)
                    break

        history.append(row)
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)

    assert best_state is not None, "No best model was saved. Check eval_every and training loop."
    model.load_state_dict(best_state)

    print("\nEvaluating on TEST set...")
    test_metrics = eval_horizons_fast(model, test_loader)
    print_metrics(f"{model_name} — TEST", test_metrics)

    save_metrics_files(run_dir, "test", test_metrics)
    append_results_summary(model_name, run_dir, test_metrics)

    save_json(run_dir / "training_summary.json", {
        "best_val_avg_MAE": best_score,
        "best_epoch": best_epoch,
        "test_metrics": test_metrics,
    })

    if save_predictions:
        pred_u, true_u = collect_selected_horizon_predictions(model, test_loader, horizons=EVAL_HORIZONS)
        np.savez_compressed(
            run_dir / "test_pred_true_selected_horizons.npz",
            pred=pred_u.astype(np.float32),   # (samples, horizons, nodes)
            true=true_u.astype(np.float32),
            horizons=np.array(EVAL_HORIZONS, dtype=np.int32),
            stations=stations,
        )

    print("\nSaved outputs to:", run_dir)
    return run_dir, test_metrics

## Classical baselines: Elastic Net and Random Forest

These two models follow the same design used in the original notebook:

- fit one model per node
- use the past 24 hours of that node as historical predictors
- append future calendar features only for the report horizons
- predict the selected horizons directly as a multi-output regression problem

This keeps the shorter-horizon study aligned with the earlier methodology.

In [5]:
H_OFF = np.array([h - 1 for h in EVAL_HORIZONS], dtype=np.int64)
H = len(EVAL_HORIZONS)


def node_features_and_targets(node: int, starts: np.ndarray):
    # This preserves the same feature engineering pattern used in the original notebook.
    Xn = X_scaled[:, node, :]  # (T, F)

    # For a 2D input with axis=0, this returns shape (T-IN_LEN+1, F, IN_LEN).
    # We keep that ordering to stay methodologically identical.
    win = np.lib.stride_tricks.sliding_window_view(Xn, window_shape=IN_LEN, axis=0)
    X_hist = win[starts].reshape(len(starts), -1)  # (S, F * IN_LEN)

    idx = starts[:, None] + IN_LEN + H_OFF[None, :]
    X_tf = TF_all[idx].reshape(len(starts), -1)    # (S, H * 4)

    X_feat = np.concatenate([X_hist, X_tf], axis=1)
    y = Y_scaled[idx, node]                        # (S, H)

    return X_feat.astype(np.float32), y.astype(np.float32)


def save_classical_run_outputs(model_name: str, run_dir: Path, pred_u: np.ndarray, true_u: np.ndarray, metrics: dict):
    save_metrics_files(run_dir, "test", metrics)
    append_results_summary(model_name, run_dir, metrics)

    np.savez_compressed(
        run_dir / "test_pred_true_selected_horizons.npz",
        pred=pred_u.astype(np.float32),   # (samples, nodes, horizons)
        true=true_u.astype(np.float32),
        horizons=np.array(EVAL_HORIZONS, dtype=np.int32),
        stations=stations,
    )

### Elastic Net

In [6]:
def run_elasticnet_baseline(alpha=1e-3, l1_ratio=0.5, jobs=4):
    model_name = "ElasticNet"
    run_dir = make_run_dir(model_name)
    print("Run dir:", run_dir)

    save_json(run_dir / "config.json", {
        "model_name": model_name,
        "alpha": alpha,
        "l1_ratio": l1_ratio,
        "jobs": jobs,
        "eval_horizons": EVAL_HORIZONS,
        "tag": SHORT_TAG,
    })

    S_test = len(test_starts)
    pred_scaled = np.zeros((S_test, N, H), dtype=np.float32)
    true_scaled = np.zeros((S_test, N, H), dtype=np.float32)

    def fit_one(node: int):
        Xtr, ytr = node_features_and_targets(node, train_starts)
        Xte, yte = node_features_and_targets(node, test_starts)

        mdl = make_pipeline(
            StandardScaler(),
            MultiTaskElasticNet(
                alpha=alpha,
                l1_ratio=l1_ratio,
                max_iter=5000,
                random_state=SEED,
            ),
        )
        mdl.fit(Xtr, ytr)
        pred = mdl.predict(Xte).astype(np.float32)
        return node, pred, yte

    results = Parallel(n_jobs=jobs, prefer="threads")(
        delayed(fit_one)(node) for node in tqdm(range(N), desc="ElasticNet per-node")
    )

    for node, pred, yte in results:
        pred_scaled[:, node, :] = pred
        true_scaled[:, node, :] = yte

    pred_u = pred_scaled * flow_std[None, :, None] + flow_mean[None, :, None]
    true_u = true_scaled * flow_std[None, :, None] + flow_mean[None, :, None]

    metrics = {}
    for j, h in enumerate(EVAL_HORIZONS):
        err = pred_u[:, :, j] - true_u[:, :, j]
        metrics[h] = {
            "MAE": float(np.abs(err).mean()),
            "RMSE": float(np.sqrt((err ** 2).mean())),
        }

    print_metrics("ElasticNet — TEST", metrics)
    save_classical_run_outputs(model_name, run_dir, pred_u, true_u, metrics)
    return run_dir, metrics


# Run Elastic Net
run_dir_en, test_en = run_elasticnet_baseline(alpha=1e-3, l1_ratio=0.5, jobs=4)
print("Elastic Net run dir:", run_dir_en)

Run dir: artifacts/runs/20260320_003509_ElasticNet_short_h1_6_12_24


ElasticNet per-node:   0%|          | 0/1821 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:2412: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 1.3010830879211426, tolerance: 0.42161405086517334
  ) = cd_fast.enet_coordinate_descent_multi_task(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:2412: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.4503946900367737, tolerance: 0.42117151618003845
  ) = cd_fast.enet_coordinate_descent_multi_task(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:2412: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 1.4267100095748901, tolerance: 0.4216039478778839
  ) = cd_fast.enet_coordinate_descent_multi_task(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:24


ElasticNet — TEST
    1h  MAE=86.664  RMSE=162.562
    6h  MAE=146.991  RMSE=296.159
   12h  MAE=150.522  RMSE=303.586
   24h  MAE=148.584  RMSE=301.620
Elastic Net run dir: artifacts/runs/20260320_003509_ElasticNet_short_h1_6_12_24


### Random Forest